# RSM from Bruker .brml, using Eq. 7 / Eq. 8 (Bruker ω / 2θ convention)

$$Q_x = \tfrac{2\pi}{\lambda}[\cos\omega - \cos(2\theta-\omega)] \qquad Q_z = \tfrac{2\pi}{\lambda}[\sin\omega + \sin(2\theta-\omega)]$$

The .brml holds the same counts as the .raw file, and they match bit for bit. It also stores in plain XML everything these equations need:
* ω and 2θ per step (`RawData0.xml` → `<Datum>`)
* the LynxEye per-channel 2θ calibration: channel *i* sits at 2θ + `ZeroOffset` + *i*·`Resolution` (`ScaleAxisInfo`)
* the frames: `MemoryMappedFiles/*.bin`, a float64 array of shape (n_steps × n_channels)

No pixel size or detector distance is needed: Bruker's calibrated angle per channel is used directly.
Equations 7/8 assume a coplanar geometry, so χ (≈ −0.6° here) is ignored. Including it moves Qz by less than 3·10⁻⁴ Å⁻¹.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [ ]:
import os, zipfile
import xml.etree.ElementTree as ET
import numpy as np

def _text(root, path, cast=float):
    el = root.find(path)
    return None if el is None else cast(el.text)

def read_bruker_brml(path, raw_index=0):
    """Read a Bruker .brml file (a zip archive of XML plus a binary detector frame file).

    Returns a dict with:
      omega, twotheta : (n_steps,) goniometer angles per step [deg] (Bruker convention)
      delta_2theta    : (n_chan,) 2theta offset of each detector channel relative to
                        the detector centre [deg], taken from the instrument calibration
      counts          : (n_steps, n_chan) intensities
      chi, phi, wavelength, plus the scan metadata
    """
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        rd_name = f"Experiment0/RawData{raw_index}.xml"
        root = ET.fromstring(z.read(rd_name))
        route = root.find(".//DataRoute")

        # per-step goniometer positions: Datum = time, absorber, Theta, 2Theta (actual),
        # Theta, 2Theta (target), 0D-mean, 0D-rms. View "ScanAxes" sits at Start=4.
        datums = np.array([[float(v) for v in d.text.split(",")]
                           for d in route.iter("Datum")])
        start = 4
        for view in route.iter("RawDataView"):
            v = view.find("Varying")
            if v is not None and v.get("LogicName") == "ScanAxes":
                start = int(view.get("Start"))
        omega, twotheta = datums[:, start], datums[:, start + 1]

        # detector channel scale: 2theta_rel(i) = ZeroOffset + i * Resolution
        scale = route.find(".//ScaleAxes/ScaleAxisInfo")
        zero_offset = float(scale.find("Start").text)
        resolution = float(scale.find("Increment").text)

        rec = route.find(".//RawDataView[@MeasuredDataIndex='0']/Recording/Size/X")
        n_chan = int(rec.text)
        delta_2theta = zero_offset + resolution * np.arange(n_chan)

        # binary frame data: n_steps x n_chan little-endian float64
        link = route.find(".//ScanDataExternalLink")
        bin_name = next(n for n in names if n.endswith(link.get("Name")))
        counts = np.frombuffer(z.read(bin_name), dtype="<f8").reshape(len(omega), n_chan)

        # fixed axes (chi, phi) and wavelength
        fixed = {}
        for d in root.findall(".//FixedInformation/Drives/InfoData"):
            pos = d.find("Position")
            if d.get("LogicName") and pos is not None:
                fixed[d.get("LogicName")] = float(pos.get("Value"))
        wl = root.find(".//WaveLength")
        wavelength = float(wl.get("Value")) if wl is not None else 1.5406

    return dict(omega=omega, twotheta=twotheta, delta_2theta=delta_2theta,
                counts=counts, chi=fixed.get("Chi", 0.0), phi=fixed.get("Phi", 0.0),
                wavelength=wavelength, resolution=resolution, zero_offset=zero_offset,
                time_per_step=datums[0, 0], name=os.path.basename(path))

def angles_to_q_bruker(omega, twotheta, delta_2theta, wavelength):
    """Eq. 7 / Eq. 8 (Bruker convention), evaluated for every (step, channel) pixel.
    Each channel sees 2theta_i = 2theta + delta_i; omega is the same for all channels."""
    k = 2 * np.pi / wavelength
    w = np.deg2rad(omega)[:, None]
    tt = np.deg2rad(twotheta[:, None] + delta_2theta[None, :])
    w = np.broadcast_to(w, tt.shape)
    Qx = k * (np.cos(w) - np.cos(tt - w))
    Qz = k * (np.sin(w) + np.sin(tt - w))
    return Qx, Qz


## Plotting

In [ ]:
def plot_rsm(Qx, Qz, I, title, flip_qx=False):
    """Qx, Qz, I are 2D (n_steps, n_channels) arrays at native resolution.
    flip_qx=True plots -Qx (matches the xrayutilities sign used in gao_sto03.1)."""
    if flip_qx:
        Qx = -Qx
    I_plot = np.where(I > 0, I, np.nan)
    fig, ax = plt.subplots(figsize=(5, 6), dpi=200)
    ax.set_facecolor("black")
    im = ax.pcolormesh(Qx, Qz, I_plot, cmap="inferno",
                       norm=LogNorm(vmin=1, vmax=np.nanmax(I)), shading="auto")
    ax.set_xlabel(("−" if flip_qx else "") + "Qx / Å⁻¹"); ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(title, fontsize=7)
    fig.colorbar(im, ax=ax, label="Intensity (counts)")
    fig.tight_layout()
    plt.show()

def plot_brml_file(path, flip_qx=False):
    d = read_bruker_brml(path)
    Qx, Qz = angles_to_q_bruker(d["omega"], d["twotheta"], d["delta_2theta"], d["wavelength"])
    plot_rsm(Qx, Qz, d["counts"], d["name"], flip_qx=flip_qx)
    return Qx, Qz, d

def plot_all_brml_in_dir(dir_path, flip_qx=False):
    files = sorted(f for f in os.listdir(dir_path) if f.lower().endswith(".brml"))
    if not files:
        print(f"no .brml files found in {dir_path}")
    for fname in files:
        print(f"--- {fname} ---")
        plot_brml_file(os.path.join(dir_path, fname), flip_qx=flip_qx)

In [ ]:
dir_1 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-15"
dir_2 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-16"
dir_3 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-29"

In [ ]:
%matplotlib inline
plt.close("all")
Qx, Qz, d = plot_brml_file(os.path.join(dir_1, "STO_013p_56.3284°-58.3484°_0.002°_3s_om_off_-36.7.brml"))

# sanity check: brightest pixel vs bulk SrTiO3 (013), a = 3.905 Å
i, j = np.unravel_index(d["counts"].argmax(), d["counts"].shape)
print(f"peak: Qx = {Qx[i, j]:.4f}, Qz = {Qz[i, j]:.4f} Å⁻¹   (STO bulk: |Qx| = {2*np.pi/3.905:.4f}, Qz = {3*2*np.pi/3.905:.4f})")

In [ ]:
%matplotlib inline
plt.close("all")
plot_all_brml_in_dir(dir_1)